In [7]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [8]:
import pandas as pd

train_df = pd.read_csv('/content/drive/MyDrive/EnviroSense/data/train.csv')
test_df = pd.read_csv('/content/drive/MyDrive/EnviroSense/data/test.csv')

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)
train_df.head()

Train shape: (800, 5)
Test shape: (200, 5)


,temperature,humidity,gas,risk_index,anomaly_label
0,29.649803,45.983566,109.592235,31.930467,0.0
1,31.366469,53.952241,120.159079,31.146189,0.0
2,22.072380,42.976139,167.089233,34.296257,0.0
3,53.793102,19.794415,407.992417,61.379226,1.0
4,37.070083,32.554757,448.239781,56.004184,1.0


## Two ML Problems

**Problem 1 — Regression:** Predict `risk_index` (a continuous 0-100 score)
from the 3 sensor inputs (`temperature`, `humidity`, `gas`).
Metric: MSE (Mean Squared Error) — measures how far off predictions are, on average.

**Problem 2 — Classification:** Predict `anomaly_label` (0 = NORMAL, 1 = ANOMALY)
from the same 3 sensor inputs.
Metrics: Accuracy, Precision, Recall, Confusion Matrix — standard classification metrics.

Both models share the same 3 inputs but solve different questions.

In [9]:
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_squared_error

# Separate inputs (X) from what we're predicting (y)
X_train = train_df[['temperature', 'humidity', 'gas']]
y_train_reg = train_df['risk_index']

X_test = test_df[['temperature', 'humidity', 'gas']]
y_test_reg = test_df['risk_index']

# Building the model: 3 inputs - 8 hidden neurons - 1 output
reg_model = MLPRegressor(hidden_layer_sizes=(8,), max_iter=1000, random_state=42)

# Training it
reg_model.fit(X_train, y_train_reg)

# Predicting on test data it has never seen
y_pred_reg = reg_model.predict(X_test)

# Measuring how good the predictions are
mse = mean_squared_error(y_test_reg, y_pred_reg)
print("MSE:", mse)
print("Sample predictions:", y_pred_reg[:5])
print("Sample actual:     ", y_test_reg[:5].values)

MSE: 8.986709457218778
Sample predictions: [32.42651867 28.16198807 27.47454423 26.64317109 24.7019893 ]
Sample actual:      [33.37689118 31.27019565 30.48871877 30.91487192 33.10589662]


In [10]:
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, confusion_matrix

# Separate inputs (X) from what we're predicting (y)
y_train_clf = train_df['anomaly_label']
y_test_clf = test_df['anomaly_label']

# Building the model: 3 inputs - 8 hidden neurons - 2-class output
clf_model = MLPClassifier(hidden_layer_sizes=(8,), max_iter=1000, random_state=42)

# Training it
clf_model.fit(X_train, y_train_clf)

# Predicting on test data
y_pred_clf = clf_model.predict(X_test)

# Measuring how good the predictions are
acc = accuracy_score(y_test_clf, y_pred_clf)
prec = precision_score(y_test_clf, y_pred_clf)
rec = recall_score(y_test_clf, y_pred_clf)
cm = confusion_matrix(y_test_clf, y_pred_clf)

print("Accuracy:", acc)
print("Precision:", prec)
print("Recall:", rec)
print("Confusion Matrix:\n", cm)

Accuracy: 0.995
Precision: 0.975609756097561
Recall: 1.0
Confusion Matrix:
 [[159   1]
 [  0  40]]


In [11]:
import joblib

joblib.dump(reg_model, '/content/drive/MyDrive/EnviroSense/models/reg_fp32.pkl')
joblib.dump(clf_model, '/content/drive/MyDrive/EnviroSense/models/clf_fp32.pkl')

print("Models saved successfully")

Models saved successfully


## Day 2 — Baseline Model Results (FP32)

**Problem 1 — Regression (risk_index prediction)**
- Architecture: 3 inputs → 8 hidden neurons → 1 output (MLPRegressor)
- Test MSE: 8.99
- Context: risk_index ranges 18-72 (spread of 54), so typical error
  is 3 points — a reasonable baseline for a small 8-neuron network.

**Problem 2 — Classification (NORMAL vs ANOMALY)**
- Architecture: 3 inputs → 8 hidden neurons → 2-class output (MLPClassifier)
- Test Accuracy: 99.5%
- Test Precision: 97.6%
- Test Recall: 100%
- Confusion Matrix: 159 true negatives, 1 false positive,
  0 false negatives, 40 true positives
- Context: High scores are expected given the clearly separated synthetic
  anomaly ranges (visually confirmed in Day 1 scatter plot). Zero missed
  anomalies (100% recall) with only one false alarm indicates a genuinely
  learned decision boundary rather than memorization.

**Models saved:** `models/reg_fp32.pkl`, `models/clf_fp32.pkl` — these are
the FP32 baselines